# كشكول التجارب: استراتيجية إعادة الترتيب (المحاضرة 44)

أهلاً بيك! في الكشكول ده هنجرب فكرة الـ `Reranking` بطريقة عملية جداً.

تخيل إنك بتعمل بحث، والمسترجع طلعلك 5 وثائق.. الوثيقة اللي فيها الإجابة بالظبط كانت في المركز الرابع أو الخامس عشان كانت مكتوبة بأسلوب مختلف شوية!  
لو بعت النتايج كده للـ LLM، ممكن مياخدش باله من الوثيقة المهمة.

هنا بقى بيجي دور الـ **Reranker**:  
بيمسك السؤال وكل وثيقة ويبص فيهم بتركيز، ويقول: *"لأ استنى، الوثيقة رقم 5 دي هي أهم واحدة هنا، اطلعي فوق في المركز الأول!"*.

تعال نشوف ده بيحصل إزاي بالكود!


In [1]:
# 1. نستورد المكتبات
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

print("المكتبات جاهزة!")


C:\Users\aliashraf\AppData\Local\Temp\ipykernel_20028\1461253270.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


المكتبات جاهزة!


### 2. شوية وثائق هنجرب عليهم
هنكتب 5 وثائق، والوثيقة رقم 4 هي اللي فيها إجابة السؤال بالظبط عن إعادة الترتيب:


In [2]:
docs = [
    Document(page_content="Python is an interpreted programming language used in web and data science.", metadata={"id": "doc1"}),
    Document(page_content="Machine learning frameworks like PyTorch help train deep neural networks.", metadata={"id": "doc2"}),
    Document(page_content="LLMs may produce hallucinations if not grounded in verified knowledge.", metadata={"id": "doc3"}),
    Document(page_content="Second-stage reranking improves RAG accuracy by reordering retrieved chunks using deep cross-attention.", metadata={"id": "doc4"}),
    Document(page_content="Flask and FastAPI are popular Python web frameworks.", metadata={"id": "doc5"})
]

print("الوثائق جاهزة!")


الوثائق جاهزة!


### 3. الاسترجاع الأولي السريع (Stage 1)
تعال نبحث في الفيكتور العادي ونشوف الترتيب المبدئي هيكون إيه:


In [3]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

query = "How does reranking improve accuracy in RAG?"
initial_results = retriever.invoke(query)

print(f"السؤال: '{query}'\n")
print("الترتيب المبدئي للوثائق من الفيكتور داتابيس:")
for rank, d in enumerate(initial_results, 1):
    print(f"  المركز {rank}: [{d.metadata['id']}] {d.page_content[:65]}...")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

السؤال: 'How does reranking improve accuracy in RAG?'

الترتيب المبدئي للوثائق من الفيكتور داتابيس:
  المركز 1: [doc4] Second-stage reranking improves RAG accuracy by reordering retrie...
  المركز 2: [doc2] Machine learning frameworks like PyTorch help train deep neural n...
  المركز 3: [doc3] LLMs may produce hallucinations if not grounded in verified knowl...
  المركز 4: [doc5] Flask and FastAPI are popular Python web frameworks....
  المركز 5: [doc1] Python is an interpreted programming language used in web and dat...


### 4. تطبيق إعادة الترتيب (Stage 2: Reranking)
دلوقتي هنخلي الموديل يمسك كل وثيقة ويديها درجة صلة حقيقية من 0 لـ 100، ونرتبهم من جديد:


In [4]:
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0)

score_prompt = ChatPromptTemplate.from_template(
    """You are a search evaluator.
Query: {query}
Document: {document}
Rate the relevance from 0 to 100. Output ONLY the integer score.
Score:"""
)

rescored = []
for doc in initial_results:
    chain = score_prompt | llm
    try:
        score_text = chain.invoke({"query": query, "document": doc.page_content}).content.strip()
        score = float("".join(c for c in score_text if c.isdigit()))
    except Exception:
        score = 0.0
    rescored.append({"doc": doc, "score": score})

# نرتب تنازلياً حسب الدرجة الجديدة
reranked = sorted(rescored, key=lambda x: x["score"], reverse=True)

print("الترتيب الجديد بعد الـ Reranker:")
for new_rank, item in enumerate(reranked, 1):
    print(f"  المركز الجديد {new_rank} [درجة: {item['score']}]: [{item['doc'].metadata['id']}] {item['doc'].page_content[:65]}...")


الترتيب الجديد بعد الـ Reranker:
  المركز الجديد 1 [درجة: 95.0]: [doc4] Second-stage reranking improves RAG accuracy by reordering retrie...
  المركز الجديد 2 [درجة: 15.0]: [doc3] LLMs may produce hallucinations if not grounded in verified knowl...
  المركز الجديد 3 [درجة: 0.0]: [doc2] Machine learning frameworks like PyTorch help train deep neural n...
  المركز الجديد 4 [درجة: 0.0]: [doc5] Flask and FastAPI are popular Python web frameworks....
  المركز الجديد 5 [درجة: 0.0]: [doc1] Python is an interpreted programming language used in web and dat...


---
### مساحتك للتجربة الحرة
لاحظت إزاي الوثيقة المهمة صعدت للمركز الأول؟  
جرب في الخلية دي تكتب سؤال مختلف وتشوف مين الوثيقة اللي هتتصدر الترتيب بعد التقييم.


In [5]:
# جرب هنا بسؤالك المفضل:
# test_q = "What are the frameworks to build web APIs in Python?"
# جرب الخطوات وشوف النتيجة
